# AR(1) con Ruido Laplace: Máxima Verosimilitud vs Mínimos Cuadrados

**Objetivo:** Estimar un modelo AR(1) cuando los errores siguen una distribución Laplace (en lugar de Normal) y comparar los resultados.

**Contenido:**
1. Simulación de datos con ruido Laplace
2. Derivación de las ecuaciones de primer orden (CPO)
3. Estimación por Máxima Verosimilitud (MV) con Laplace
4. Estimación por Programación Lineal (LAD)
5. Comparación con MCO (que asume Normalidad)
6. Visualización y diagnóstico

## 1. Marco Teórico

### Modelo AR(1)

$$y_t = \mu + \rho y_{t-1} + \varepsilon_t$$

### Distribución Laplace

Si $\varepsilon_t \sim \text{Laplace}(0, b)$, su densidad es:

$$f(\varepsilon_t) = \frac{1}{2b} \exp\left(-\frac{|\varepsilon_t|}{b}\right)$$

### Log-verosimilitud condicional

$$\ell(\mu, \rho, b) = -(T-1)\ln(2b) - \frac{1}{b} \sum_{t=2}^{T} |y_t - \mu - \rho y_{t-1}|$$

### Ecuaciones de primer orden (CPO)

**Para $\mu$:**
$$\frac{\partial \ell}{\partial \mu} = \frac{1}{b} \sum_{t=2}^{T} \text{sgn}(y_t - \mu - \rho y_{t-1}) = 0$$

**Para $\rho$:**
$$\frac{\partial \ell}{\partial \rho} = \frac{1}{b} \sum_{t=2}^{T} y_{t-1} \cdot \text{sgn}(y_t - \mu - \rho y_{t-1}) = 0$$

**Para $b$:**
$$\hat{b} = \frac{1}{T-1} \sum_{t=2}^{T} |y_t - \mu - \rho y_{t-1}|$$

**Conclusión:** Las dos primeras ecuaciones son **no lineales** (por el signo) y **no tienen solución cerrada**. Se resuelven por métodos numéricos o programación lineal.

In [3]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linprog
from scipy.stats import norm, probplot

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Semilla para reproducibilidad
np.random.seed(42)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Simulación de datos

Generamos una serie AR(1) con ruido Laplace usando la **transformada inversa**:

$$X = b \cdot \ln\left(\frac{U_1}{U_2}\right)$$

donde $U_1, U_2 \sim \text{Uniforme}(0,1)$.

In [ ]:
# ============================================================
# SIMULACIÓN DE DATOS
# ============================================================
T = 500
mu_true = 2.0
rho_true = 0.7
b_true = 0.5  # parámetro de escala de Laplace

# Generar ruido Laplace usando transformada inversa
u1 = np.random.uniform(0, 1, T)
u2 = np.random.uniform(0, 1, T)
epsilon = b_true * np.log(u1 / u2)

# Generar serie AR(1)
y = np.zeros(T)
y[0] = mu_true / (1 - rho_true)  # condición inicial = media

for t in range(1, T):
    y[t] = mu_true + rho_true * y[t-1] + epsilon[t]

print(f"Serie simulada con T = {T} observaciones")
print(f"Media teórica: {mu_true / (1 - rho_true):.4f}")
print(f"Media muestral: {np.mean(y):.4f}")
print(f"Desviación estándar muestral: {np.std(y):.4f}")

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LA SERIE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Serie temporal
axes[0].plot(y, color='steelblue', linewidth=0.8)
axes[0].axhline(y=mu_true/(1-rho_true), color='red', linestyle='--', 
                label=f'Media = {mu_true/(1-rho_true):.2f}')
axes[0].set_title('Serie AR(1) simulada con ruido Laplace')
axes[0].set_xlabel('t')
axes[0].set_ylabel('y_t')
axes[0].legend()

# Histograma del ruido
axes[1].hist(epsilon, bins=40, density=True, alpha=0.7, 
             color='steelblue', edgecolor='black', label='Ruido simulado')
x_vals = np.linspace(epsilon.min(), epsilon.max(), 200)
laplace_pdf = (1/(2*b_true)) * np.exp(-np.abs(x_vals)/b_true)
axes[1].plot(x_vals, laplace_pdf, 'r-', linewidth=2, label='Laplace teórica')
axes[1].set_title('Distribución del ruido')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Función de Log-Verosimilitud Negativa

Para usar `scipy.optimize.minimize`, definimos la **log-verosimilitud negativa**:

$$-\ell = n \ln(2b) + \frac{1}{b} \sum_{t=2}^{T} |y_t - \mu - \rho y_{t-1}|$$

donde $n = T-1$.

In [ ]:
# ============================================================
# FUNCIÓN DE LOG-VEROSIMILITUD NEGATIVA
# ============================================================
def neg_log_likelihood(params, y, with_intercept=True):
    """
    Calcula la log-verosimilitud negativa para un AR(1) con ruido Laplace.
    
    Parámetros:
    -----------
    params : array
        Si with_intercept=True: [mu, rho, b]
        Si with_intercept=False: [rho, b]
    y : array
        Serie de tiempo
    with_intercept : bool
        Si True, incluye intercepto
    
    Retorna:
    --------
    float : log-verosimilitud negativa
    """
    if with_intercept:
        mu, rho, b = params
        residuals = y[1:] - mu - rho * y[:-1]
    else:
        rho, b = params
        residuals = y[1:] - rho * y[:-1]
    
    n = len(residuals)
    
    # Penalización si b <= 0
    if b <= 0:
        return 1e10
    
    # -ell = n*ln(2b) + (1/b)*sum|res|
    return n * np.log(2 * b) + np.sum(np.abs(residuals)) / b

print("Función definida correctamente.")

## 4. Estimación con Intercepto (Método Numérico)

Usamos `Nelder-Mead` porque la función de valor absoluto no es diferenciable en cero.

In [ ]:
# ============================================================
# ESTIMACIÓN CON INTERCEPTO
# ============================================================
x0 = [1.0, 0.5, 1.0]  # mu, rho, b

result_intercept = minimize(
    neg_log_likelihood, 
    x0, 
    args=(y, True),
    method='Nelder-Mead',
    options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8}
)

mu_hat, rho_hat, b_hat = result_intercept.x

print("=" * 60)
print("ESTIMACIÓN CON INTERCEPTO (Laplace)")
print("=" * 60)
print(f"μ̂ = {mu_hat:.4f}  (verdadero: {mu_true})")
print(f"ρ̂ = {rho_hat:.4f}  (verdadero: {rho_true})")
print(f"b̂ = {b_hat:.4f}  (verdadero: {b_true})")
print(f"Log-verosimilitud negativa: {result_intercept.fun:.4f}")
print(f"Convergencia: {result_intercept.success}")

## 5. Estimación sin Intercepto (Método Numérico)

In [ ]:
# ============================================================
# ESTIMACIÓN SIN INTERCEPTO
# ============================================================
x0_no = [0.5, 1.0]  # rho, b

result_no_intercept = minimize(
    neg_log_likelihood, 
    x0_no, 
    args=(y, False),
    method='Nelder-Mead',
    options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8}
)

rho_hat_no, b_hat_no = result_no_intercept.x

print("=" * 60)
print("ESTIMACIÓN SIN INTERCEPTO (Laplace)")
print("=" * 60)
print(f"ρ̂ = {rho_hat_no:.4f}  (verdadero: {rho_true})")
print(f"b̂ = {b_hat_no:.4f}  (verdadero: {b_true})")
print(f"Log-verosimilitud negativa: {result_no_intercept.fun:.4f}")
print(f"Convergencia: {result_no_intercept.success}")

## 6. Estimación por Programación Lineal (LAD)

Minimizar $\sum |e_t|$ es equivalente a un problema de programación lineal:

$$\min \sum_{t=2}^{T} u_t$$

sujeto a:
$$u_t \geq y_t - \mu - \rho y_{t-1}$$
$$u_t \geq -(y_t - \mu - \rho y_{t-1})$$
$$u_t \geq 0$$

In [ ]:
# ============================================================
# ESTIMACIÓN POR PROGRAMACIÓN LINEAL (LAD)
# ============================================================
def lad_estimation(y):
    """
    Estima un AR(1) por LAD usando programación lineal.
    Minimiza sum|y_t - mu - rho*y_{t-1}|
    """
    n = len(y) - 1
    # Variables: [mu, rho, u_1, ..., u_n]
    c = np.zeros(2 + n)
    c[2:] = 1.0  # Coeficientes de u_i en la función objetivo
    
    # Restricciones: A_ub @ x <= b_ub
    A_ub = np.zeros((2 * n, 2 + n))
    b_ub = np.zeros(2 * n)
    
    for i in range(n):
        # Restricción 1: -mu - rho*y_i - u_i <= -y_{i+1}
        A_ub[2*i, 0] = -1
        A_ub[2*i, 1] = -y[i]
        A_ub[2*i, 2+i] = -1
        b_ub[2*i] = -y[i+1]
        
        # Restricción 2: mu + rho*y_i - u_i <= y_{i+1}
        A_ub[2*i+1, 0] = 1
        A_ub[2*i+1, 1] = y[i]
        A_ub[2*i+1, 2+i] = -1
        b_ub[2*i+1] = y[i+1]
    
    # Límites: mu, rho libres; u_i >= 0
    bounds = [(None, None), (None, None)] + [(0, None)] * n
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    
    if res.success:
        return res.x[0], res.x[1]
    else:
        return None, None

mu_lad, rho_lad = lad_estimation(y)

print("=" * 60)
print("ESTIMACIÓN POR PROGRAMACIÓN LINEAL (LAD)")
print("=" * 60)
print(f"μ̂ = {mu_lad:.4f}  (verdadero: {mu_true})")
print(f"ρ̂ = {rho_lad:.4f}  (verdadero: {rho_true})")

## 7. Comparación con MCO (que asume Normalidad)

El estimador MCO minimiza $\sum (y_t - \mu - \rho y_{t-1})^2$.

In [ ]:
# ============================================================
# ESTIMACIÓN POR MCO
# ============================================================
X = np.column_stack([np.ones(T-1), y[:-1]])
beta_ols = np.linalg.lstsq(X, y[1:], rcond=None)[0]

print("=" * 60)
print("COMPARACIÓN: MLE-Laplace vs MCO-Normal")
print("=" * 60)
print(f"MCO:      μ̂ = {beta_ols[0]:.4f}, ρ̂ = {beta_ols[1]:.4f}")
print(f"Laplace:  μ̂ = {mu_hat:.4f}, ρ̂ = {rho_hat:.4f}")
print(f"LAD (LP): μ̂ = {mu_lad:.4f}, ρ̂ = {rho_lad:.4f}")
print(f"Verdadero: μ = {mu_true}, ρ = {rho_true}")

## 8. Visualización de Resultados

In [ ]:
# ============================================================
# VISUALIZACIÓN COMPLETA
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel 1: Serie simulada ---
axes[0, 0].plot(y, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(y=mu_true/(1-rho_true), color='red', linestyle='--', 
                    label=f'Media = {mu_true/(1-rho_true):.2f}')
axes[0, 0].set_title('Serie AR(1) con ruido Laplace')
axes[0, 0].set_xlabel('t')
axes[0, 0].set_ylabel('y_t')
axes[0, 0].legend()

# --- Panel 2: Histograma de residuos vs Laplace y Normal ---
residuals = y[1:] - mu_hat - rho_hat * y[:-1]
axes[0, 1].hist(residuals, bins=40, density=True, alpha=0.7, 
                color='steelblue', edgecolor='black', label='Residuos')
x_vals = np.linspace(residuals.min(), residuals.max(), 200)

# Laplace ajustada
laplace_pdf = (1/(2*b_hat)) * np.exp(-np.abs(x_vals)/b_hat)
axes[0, 1].plot(x_vals, laplace_pdf, 'r-', linewidth=2, label='Laplace ajustada')

# Normal ajustada
normal_pdf = norm.pdf(x_vals, 0, np.std(residuals))
axes[0, 1].plot(x_vals, normal_pdf, 'g--', linewidth=2, label='Normal ajustada')

axes[0, 1].set_title('Distribución de residuos')
axes[0, 1].legend()

# --- Panel 3: QQ-plot vs Normal ---
probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('QQ-plot vs Normal')

# --- Panel 4: Comparación de estimadores ---
estimadores = ['μ', 'ρ', 'b']
valores_true = [mu_true, rho_true, b_true]
valores_mle = [mu_hat, rho_hat, b_hat]

x_pos = np.arange(len(estimadores))
width = 0.35

axes[1, 1].bar(x_pos - width/2, valores_true, width, 
               label='Verdadero', color='gray', alpha=0.7)
axes[1, 1].bar(x_pos + width/2, valores_mle, width, 
               label='MLE Laplace', color='steelblue', alpha=0.7)
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(estimadores)
axes[1, 1].set_title('Comparación de estimadores')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 9. Conclusiones

1. **Con Normalidad:** El estimador MV es MCO. Solución cerrada.
2. **Con Laplace:** El estimador MV es LAD (mínimas desviaciones absolutas). **No hay solución cerrada.**
3. **Método de solución:** Programación lineal o optimización numérica (Nelder-Mead).
4. **Diferencia clave:** LAD es más robusto a outliers porque la distribución Laplace tiene colas más pesadas.
5. **Conexión:** AR(1) con ruido Laplace = regresión LAD de $y_t$ sobre $y_{t-1}$.

### Referencias

- Wikipedia: Distribución de Laplace
- Norton, R.M. (1984). "The Double Exponential Distribution: Using Calculus to Find a Maximum Likelihood Estimator". The American Statistician.

In [ ]:
# ============================================================
# VERIFICACIÓN FINAL
# ============================================================
print("=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"\nModelo: y_t = μ + ρ y_(t-1) + ε_t, ε_t ~ Laplace(0, b)")
print(f"\nParámetros verdaderos: μ = {mu_true}, ρ = {rho_true}, b = {b_true}")
print(f"\nEstimación MLE (numérica): μ̂ = {mu_hat:.4f}, ρ̂ = {rho_hat:.4f}, b̂ = {b_hat:.4f}")
print(f"Estimación LAD (LP):       μ̂ = {mu_lad:.4f}, ρ̂ = {rho_lad:.4f}")
print(f"Estimación MCO (Normal):   μ̂ = {beta_ols[0]:.4f}, ρ̂ = {beta_ols[1]:.4f}")
print("\n" + "=" * 60)
print("Conclusión: MLE con Laplace = LAD. MCO es inconsistente si los errores son Laplace.")
print("=" * 60)